In [1]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys
from collections import Counter

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [2]:
# Scripts 
from src.local_config import *
from src.portfolio_con import *

In [3]:
# Load in data from previous notebook
df1_is = pd.read_csv(PROJECT_ROOT / "data/df1_is",
                     index_col=0, parse_dates=True)
df1_oos = pd.read_csv(PROJECT_ROOT / "data/df1_oos",
                      index_col=0, parse_dates=True)

df2_is = pd.read_csv(PROJECT_ROOT / "data/df2_is",
                     index_col=0, parse_dates=True)
df2_oos = pd.read_csv(PROJECT_ROOT / "data/df2_oos",
                      index_col=0, parse_dates=True)

# Portfolio Construction

- Test for correlation when combining our pairs for trading to ensure that the portfolio is `diversified'.
- Then give a 50/50 weighting to each sector to ensure fairness in the portfolio when trading

## Correlation between pairs

Only do diagnostics on in-sample and then use the same pairs for out-of-sample portfolio construction (to prevent look-ahead bias from occurring).

In [4]:
# Load in results from previous notebook
static_results_df_is = pd.read_csv(PROJECT_ROOT / "data/static_hedge_ratio_is")
dynamic_results_df_is = pd.read_csv(PROJECT_ROOT / "data/dynamic_hedge_ratio_is")

In [5]:
# Dynamically-identified pairs from the cointegration notebook (nb2).
with open(PROJECT_ROOT / "data/selected_pairs.pkl", "rb") as f:
    selected_pairs = pickle.load(f)

_df_is = {"tech": df1_is, "commodity": df2_is}
pairs_is = [
    {"y": p["y"], "x": p["x"], "df": _df_is[p["sector"]]}
    for p in selected_pairs
]
pairs_is

[{'y': '285 HK Equity',
  'x': '3690 HK Equity',
  'df':             1211 HK Equity  1347 HK Equity  1810 HK Equity  2382 HK Equity  \
  Date                                                                         
  2015-01-01        2.314217        2.326302             NaN        2.593761   
  2015-01-02        2.359249        2.284421             NaN        2.580217   
  2015-01-05        2.359249        2.271094             NaN        2.589267   
  2015-01-06        2.363962        2.297573             NaN        2.578701   
  2015-01-07        2.275522        2.287471             NaN        2.532108   
  ...                    ...             ...             ...             ...   
  2022-09-26        4.264551        2.967333        2.269028        4.396915   
  2022-09-27        4.268298        2.964242        2.271094        4.375128   
  2022-09-28        4.224393        2.917771        2.233235        4.347047   
  2022-09-29        4.203692        2.905808        2.183802     

In [6]:
spread_returns_dict = {}

for _, row in static_results_df_is.iterrows():
    pair   = row["pair"]
    y_name, x_name = [s.strip() for s in pair.split(" vs ")]
    df     = df1_is if y_name in df1_is.columns else df2_is
    spread = df[y_name] - row["beta"] * df[x_name]
    spread_returns_dict[pair] = spread.diff().dropna()

results = portfolio_diagnostics(spread_returns_dict)

In [7]:
print(results["correlation"].round(3))
print(results["initial_vif"].round(2))
print(results["final_vif"].round(2))
print("Retained:     ", results["retained_pairs"])
print("Corr dropped: ", results["corr_dropped"])
print("VIF dropped:  ", results["vif_dropped"])

                                  285 HK Equity vs 3690 HK Equity  \
285 HK Equity vs 3690 HK Equity                             1.000   
1818 HK Equity vs 1921 HK Equity                           -0.006   
2386 HK Equity vs 386 HK Equity                             0.023   

                                  1818 HK Equity vs 1921 HK Equity  \
285 HK Equity vs 3690 HK Equity                             -0.006   
1818 HK Equity vs 1921 HK Equity                             1.000   
2386 HK Equity vs 386 HK Equity                             -0.035   

                                  2386 HK Equity vs 386 HK Equity  
285 HK Equity vs 3690 HK Equity                             0.023  
1818 HK Equity vs 1921 HK Equity                           -0.035  
2386 HK Equity vs 386 HK Equity                             1.000  
285 HK Equity vs 3690 HK Equity     1.0
1818 HK Equity vs 1921 HK Equity    1.0
2386 HK Equity vs 386 HK Equity     1.0
Name: VIF, dtype: float64
285 HK Equity vs 3690 HK

In [8]:
# Build IS/OOS portfolio pair lists from the pairs RETAINED after the
# correlation/VIF diversification filter above (results["retained_pairs"]),
# so the portfolio always reflects the dynamically-selected, diversified set.
retained = list(results["retained_pairs"])

def _to_pair_dicts(df_map):
    out = []
    for q in retained:
        y_name, x_name = [s.strip() for s in q.split(" vs ")]
        sector = "tech" if y_name in df1_is.columns else "commodity"
        out.append({"y": y_name, "x": x_name, "df": df_map[sector]})
    return out

portfolio_is  = _to_pair_dicts({"tech": df1_is,  "commodity": df2_is})
portfolio_oos = _to_pair_dicts({"tech": df1_oos, "commodity": df2_oos})

with open(PROJECT_ROOT / "data/portfolio_is.pkl", "wb") as f:
    pickle.dump(portfolio_is, f)

with open(PROJECT_ROOT / "data/portfolio_oos.pkl", "wb") as f:
    pickle.dump(portfolio_oos, f)

portfolio_is

[{'y': '285 HK Equity',
  'x': '3690 HK Equity',
  'df':             1211 HK Equity  1347 HK Equity  1810 HK Equity  2382 HK Equity  \
  Date                                                                         
  2015-01-01        2.314217        2.326302             NaN        2.593761   
  2015-01-02        2.359249        2.284421             NaN        2.580217   
  2015-01-05        2.359249        2.271094             NaN        2.589267   
  2015-01-06        2.363962        2.297573             NaN        2.578701   
  2015-01-07        2.275522        2.287471             NaN        2.532108   
  ...                    ...             ...             ...             ...   
  2022-09-26        4.264551        2.967333        2.269028        4.396915   
  2022-09-27        4.268298        2.964242        2.271094        4.375128   
  2022-09-28        4.224393        2.917771        2.233235        4.347047   
  2022-09-29        4.203692        2.905808        2.183802     

In [9]:
# 50/50 sector weighting, computed dynamically for whatever pairs were retained.
# Each sector present receives an equal share of capital (1 / n_sectors), split
# equally across the retained pairs within that sector. With 1 tech + 3 commodity
# pairs this reproduces tech=0.50 and each commodity=1/6.
retained = list(results["retained_pairs"])
pair_sector = {
    q: ("tech" if q.split(" vs ")[0].strip() in df1_is.columns else "commodity")
    for q in retained
}
sector_counts = Counter(pair_sector.values())
n_sectors = len(sector_counts)

portfolio = pd.DataFrame([
    {
        "pair": q,
        "sector": pair_sector[q],
        "weight": (1.0 / n_sectors) / sector_counts[pair_sector[q]],
    }
    for q in retained
])

portfolio = portfolio.merge(
    static_results_df_is[["pair", "beta"]],
    on="pair"
)

portfolio = portfolio.rename(columns={"beta": "hedge_ratio"})

portfolio.to_csv(PROJECT_ROOT / "data/portfolio", index=False)
portfolio

,pair,sector,weight,hedge_ratio
0,285 HK Equity vs 3690 HK Equity,tech,0.50,0.791044
1,1818 HK Equity vs 1921 HK Equity,commodity,0.25,-0.327249
2,2386 HK Equity vs 386 HK Equity,commodity,0.25,1.174470
